# Certification — `STL_Healpix_Kernel_Torch` : parité d'interface avec le noyau 2D

Ce notebook certifie les **étapes 1 à 3** du portage HEALPix :

1. `Base_DataClass` et `ST_Operator` rendus indépendants du type de données
   (`NDIM_PIX`, `_infer_N0`, indexations sans axes de pixels codés en dur) ;
2. `STL_Healpix_Kernel_Torch` dérivée de `Base_DataClass`
   (`pbc`, `dg`, `N0`, `conv_history`, `cell_ids`, `divide`, `get_ST_op`) ;
3. les statistiques remontées sur l'opérateur ondelettes
   (`mean`, `square_mean`, `cov`, `standardize`, `unstandardize`,
   `_compute_and_store_cross_cov`, `j_to_dg`, `mask_full_res`, `downsample`).

**Hors périmètre à ce stade** (voir la fin du notebook) :

* `get_CS_op()` — sur la sphère le spectre de puissance est `anafast`, pas un
  binning FFT. Tant qu'il n'est pas écrit, on construit l'opérateur avec
  `compute_PS=False` (c'est le défaut de `get_ST_op` pour ce type de données).
* la gestion complète des masques / NaN (`mask_full_res`), dont l'équivalent
  sphérique est l'érosion du masque de validité par le support du stencil.

In [1]:
import os
import sys
import warnings

import numpy as np
import torch
import healpy as hp
import matplotlib.pyplot as plt

# --- remonter jusqu'à la racine du dépôt (le dossier qui contient STL_main) ---
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "STL_main")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("Répertoire STL_main introuvable au-dessus de %s" % os.getcwd())
    ROOT = parent
sys.path.insert(0, ROOT)

DATA_TEST_PATH = os.path.join(ROOT, "data", "test")
print("Racine du dépôt :", ROOT)

warnings.filterwarnings("ignore", category=UserWarning)

from STL_main.ST_Operator import ST_Operator
from STL_main.STL_Healpix_Kernel_Torch import (
    STL_Healpix_Kernel_Torch as DataClass,
    WaveletOperatorHealpixKernel_torch,
)

torch.manual_seed(0)
np.random.seed(0)

Racine du dépôt : /home/claude/work/STL-Dev


## 0. Carte de test

On prend la carte LSS du dépôt si elle est présente, sinon une réalisation
gaussienne d'un spectre en loi de puissance. Ordre **NESTED** dans les deux cas
(le sous-échantillonnage HEALPix l'exige).

In [2]:
NSIDE = 32
NPIX = 12 * NSIDE**2

lss_file = os.path.join(DATA_TEST_PATH, "Test_Heal_LSS.npy")
if os.path.exists(lss_file):
    heal_im = np.load(lss_file)
    heal_im = np.mean(heal_im.reshape(NPIX, heal_im.shape[0] // NPIX), 1)
    origin = "Test_Heal_LSS.npy"
else:
    ell = np.arange(3 * NSIDE)
    cl = 1.0 / (ell + 10.0) ** 2.5
    heal_im = hp.reorder(hp.synfast(cl, NSIDE), r2n=True)
    origin = "synfast (loi de puissance)"

heal_im = (heal_im - heal_im.mean()) / heal_im.std()
print("carte :", origin, "| nside =", NSIDE, "| Npix =", heal_im.shape)

hp.mollview(heal_im, nest=True, cmap="plasma", title="Carte de test (NESTED)")
plt.show()

carte : synfast (loi de puissance) | nside = 32 | Npix = (12288,)


## 1. La classe de données hérite bien de `Base_DataClass`

Les métadonnées attendues par le code indépendant du type de données doivent
toutes être présentes : `DT`, `NDIM_PIX`, `N0`, `dg`, `pbc`, `conv_history`,
plus les champs propres à HEALPix `cell_ids` / `nest` / `nside`.

In [3]:
from STL_main.Base_DataClass import Base_DataClass

data = DataClass(heal_im)

assert isinstance(data, Base_DataClass), "la classe doit dériver de Base_DataClass"
assert data.NDIM_PIX == 1
assert data.N0 == (NSIDE,), data.N0
assert len(data.N0) == data.NDIM_PIX, "len(N0) doit valoir NDIM_PIX"
assert data.dg == 0 and data.nside == NSIDE
assert data.pbc is True, "ciel complet -> pbc True"
assert data.conv_history == []
assert data.cell_ids.shape == (NPIX,)

print("DT           :", data.DT)
print("N0 / dg      :", data.N0, "/", data.dg)
print("nside actuel :", data.nside)
print("pbc          :", data.pbc)
print("device/dtype :", data.device, data.dtype)
print("cell_ids     :", data.cell_ids[:5].tolist(), "...")

DT           : HealpixKernel_torch
N0 / dg      : (32,) / 0
nside actuel : 32
pbc          : True
device/dtype : cpu torch.float64
cell_ids     : [0, 1, 2, 3, 4] ...


### `copy`, `__getitem__`, `modulus`, `divide`

`divide` est indispensable aux termes S1 croisés (`I*psi / |I*psi|^0.5`) ; il
manquait complètement à la version précédente.

In [4]:
batch = DataClass(np.stack([heal_im, -heal_im]))       # (2, Npix)

sub = batch[0]                                          # slicing sur les dims de tête
assert sub.array.shape == (NPIX,)
assert sub.cell_ids.shape == (NPIX,)

cp = batch.copy()
cp.array[:] = 0.0
assert not torch.allclose(cp.array, batch.array), "copy() doit être profonde sur array"

mod = batch.modulus()
assert torch.allclose(mod.array, batch.array.abs())

num = DataClass(np.ones((2, NPIX)))
den = DataClass(4.0 * np.ones((2, NPIX)))
div = num.divide(den, epsilon=0.0, pow=0.5)
assert torch.allclose(div.array, torch.full_like(div.array, 0.5))

print("copy / __getitem__ / modulus / divide : OK")

copy / __getitem__ / modulus / divide : OK


## 2. Opérateur ondelettes : convolution sphérique

`apply(data, j)` renvoie `[..., L, Npix]` et renseigne désormais
`conv_history`, ce dont dépend le choix du masque par couche.

In [5]:
J, L, KERNELSZ = 4, 4, 5
wav_op = data.get_wavelet_op(J=J, L=L, kernel_size=KERNELSZ)

print("J =", wav_op.J, "| L =", wav_op.L, "| KERNELSZ =", wav_op.KERNELSZ)
print("j_to_dg =", list(wav_op.j_to_dg))
print("mask_full_res =", wav_op.mask_full_res)

conv = wav_op.apply(data, 0)
assert conv.array.shape == (L, NPIX)
assert conv.conv_history == [0], conv.conv_history
assert conv.dg == 0 and conv.nside == NSIDE
assert torch.is_complex(conv.array)

fig = plt.figure(figsize=(16, 4))
for k in range(L):
    hp.mollview(
        conv.array[k].abs().cpu().numpy(),
        nest=True, hold=False, sub=(1, L, 1 + k),
        title=r"$|I*\psi|$ orientation %d" % k, cmap="viridis",
    )
plt.show()

J = 4 | L = 4 | KERNELSZ = 5
j_to_dg = [0, 1, 2, 3]
mask_full_res = None


### Seconde convolution et historique

La chaîne `|I*psi_2| * psi_3` doit produire `[..., L2, L3, Npix]` — c'est ce
que `ST_Operator` attend pour S3 et S4.

In [6]:
mod1 = conv.modulus()
conv2 = wav_op.apply(mod1, 0)

assert conv2.array.shape == (L, L, NPIX), conv2.array.shape
assert conv2.conv_history == [0, 0]
print("couche 1 :", tuple(conv.array.shape), conv.conv_history)
print("couche 2 :", tuple(conv2.array.shape), conv2.conv_history)

couche 1 : (4, 12288) [0]
couche 2 : (4, 4, 12288) [0, 0]


## 3. Sous-échantillonnage

Signature alignée sur le noyau planaire : `downsample(data, dg_out,
inplace=..., replace_nan_value=...)`. Lissage passe-bas gaussien sur la sphère,
puis moyenne des 4 pixels enfants en NESTED.

In [7]:
cur = DataClass(heal_im)
print("dg=0 : Npix =", cur.array.shape[-1], "| nside =", cur.nside)

fig = plt.figure(figsize=(16, 4))
hp.mollview(cur.array.cpu().numpy(), nest=True, hold=False, sub=(1, J, 1),
            title="dg = 0", cmap="plasma")
for dg in range(1, J):
    cur = wav_op.downsample(cur, dg, inplace=False)
    assert cur.dg == dg
    assert cur.nside == NSIDE // 2**dg
    assert cur.array.shape[-1] == 12 * cur.nside**2
    assert cur.cell_ids.shape[-1] == cur.array.shape[-1]
    hp.mollview(cur.array.cpu().numpy(), nest=True, hold=False, sub=(1, J, 1 + dg),
                title="dg = %d (nside %d)" % (dg, cur.nside), cmap="plasma")
plt.show()
print("sous-échantillonnage : OK")

dg=0 : Npix = 12288 | nside = 32


sous-échantillonnage : OK


## 4. Statistiques portées sur l'opérateur

C'était le cœur de la divergence : `mean`, `square_mean` et `cov` vivaient sur
la classe de données avec des signatures incompatibles. Elles sont désormais
sur l'opérateur, exactement comme dans `STL_2D_Kernel_Torch`.

In [8]:
x = DataClass(np.stack([heal_im, 2.0 * heal_im]))   # (2, Npix)

m = wav_op.mean(x)
s2 = wav_op.square_mean(x)
c = wav_op.cov(x, x)

assert m.shape == (2,) and s2.shape == (2,) and c.shape == (2,)
assert torch.allclose(m, x.array.mean(dim=-1))
assert torch.allclose(s2.real, (x.array**2).mean(dim=-1))
assert torch.allclose(c.real, s2.real)

print("mean        =", m.cpu().numpy())
print("square_mean =", s2.real.cpu().numpy())
print("cov(x, x)   =", c.real.cpu().numpy())

mean        = [3.45499092e-17 6.90998185e-17]
square_mean = [1. 4.]
cov(x, x)   = [1. 4.]


In [9]:
# standardize / unstandardize : aller-retour exact
std_data, mean_used, std_used = wav_op.standardize(x, mean_field=False, inplace=False)

assert torch.allclose(wav_op.mean(std_data).real, torch.zeros(2, dtype=std_data.array.dtype), atol=1e-10)
assert torch.allclose(wav_op.cov(std_data, std_data).real, torch.ones(2, dtype=std_data.array.dtype), atol=1e-10)

back = wav_op.unstandardize(std_data, mean_used, std_used, inplace=False)
err = (back.array - x.array).abs().max().item()
assert err < 1e-10, err

print("standardize : moyenne nulle, variance unité — aller-retour à %.2e près" % err)

standardize : moyenne nulle, variance unité — aller-retour à 0.00e+00 près


## 5. Chaîne complète `ST_Operator` : S1 à S4

Le test décisif : le même code indépendant du type de données que pour le
planaire, appliqué à des données HEALPix.

In [10]:
st_op = data.get_ST_op(J=J, L=L)
print("compute_PS =", st_op.compute_PS, "(anafast pas encore branché)")

st = st_op.apply(data, norm="vanilla")   # sans normalisation : S2 vaudrait 1 partout

expected = {
    "S1": (1, 1, 1, J, L),
    "S2": (1, 1, 1, J, L),
    "S3": (1, 1, 1, J, J, L, L),
    "S4": (1, 1, 1, J, J, J, L, L, L),
}
for name, shape in expected.items():
    got = tuple(getattr(st, name).shape)
    assert got == shape, (name, got, shape)
    print("%-3s %-28s  fraction finie = %.3f" % (
        name, str(got), torch.isfinite(getattr(st, name)).float().mean().item()))

print("mean =", st.mean.real.cpu().numpy(), "| var =", st.var.real.cpu().numpy())

compute_PS = False (anafast pas encore branché)


S1  (1, 1, 1, 4, 4)               fraction finie = 1.000
S2  (1, 1, 1, 4, 4)               fraction finie = 1.000
S3  (1, 1, 1, 4, 4, 4, 4)         fraction finie = 0.625
S4  (1, 1, 1, 4, 4, 4, 4, 4, 4)   fraction finie = 0.312
mean = [[3.45499092e-17]] | var = [[1.]]


In [11]:
plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
for lidx in range(L):
    plt.plot(np.arange(J), st.S1[0, 0, 0, :, lidx].real.cpu().numpy(), marker="o",
             label="orientation %d" % lidx)
plt.yscale("log"); plt.xlabel("j"); plt.ylabel("S1"); plt.title("S1(j, theta)")
plt.legend(fontsize=8); plt.grid(alpha=.3)

plt.subplot(1, 3, 2)
for lidx in range(L):
    plt.plot(np.arange(J), st.S2[0, 0, 0, :, lidx].real.cpu().numpy(), marker="o")
plt.yscale("log"); plt.xlabel("j"); plt.ylabel("S2"); plt.title("S2(j, theta)")
plt.grid(alpha=.3)

plt.subplot(1, 3, 3)
s4 = st.S4.abs().cpu().numpy().flatten()
plt.plot(s4[np.isfinite(s4)], lw=.8)
plt.yscale("log"); plt.title("|S4| (coefficients calculés)"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Batch, canaux multiples et statistiques croisées

In [12]:
Nb, Nc = 2, 2
maps = np.stack([np.stack([heal_im, np.roll(heal_im, 137)]) for _ in range(Nb)])
maps = maps + 0.05 * np.random.randn(*maps.shape)
multi = DataClass(maps)                                   # (Nb, Nc, Npix)

cross = torch.triu(torch.ones((Nc, Nc), dtype=bool))
st_multi = st_op.apply(multi, compute_cross_matrix=cross, norm="vanilla")

assert tuple(st_multi.S4.shape) == (Nb, Nc, Nc, J, J, J, L, L, L)
print("S1", tuple(st_multi.S1.shape))
print("S4", tuple(st_multi.S4.shape))
print("fraction finie S1 = %.2f (la sous-diagonale canaux reste NaN, comme en 2D)"
      % torch.isfinite(st_multi.S1).float().mean().item())

S1 (2, 2, 2, 4, 4)
S4 (2, 2, 2, 4, 4, 4, 4, 4, 4)
fraction finie S1 = 0.75 (la sous-diagonale canaux reste NaN, comme en 2D)


### Normalisation `store_ref` / `load_ref` et standardisation

In [13]:
st_ref = st_op.apply(multi, compute_cross_matrix=cross, standardize=True, norm="store_ref")
assert st_ref.norm and st_ref.standardized

other = DataClass(maps + 0.3 * np.random.randn(*maps.shape))
st_load = st_op.apply(other, compute_cross_matrix=cross, norm="load_ref")
assert st_load.norm

st_iso = st_op.apply(multi, compute_cross_matrix=cross, norm="vanilla", iso=True)
print("store_ref / load_ref / iso : OK")
print("S3 isotropisé :", tuple(st_iso.S3.shape))

store_ref / load_ref / iso : OK
S3 isotropisé : (2, 2, 2, 4, 4, 4)


## 6. Différentiabilité

La synthèse repose entièrement sur `backward()` : le gradient doit remonter
jusqu'à la carte d'entrée à travers le stencil sphérique.

In [14]:
u = torch.tensor(heal_im, dtype=torch.float64, requires_grad=True)
st_u = st_op.apply(DataClass(u), norm="vanilla")

loss = torch.nansum(st_u.S2.real) + st_u.var.real.sum()
loss.backward()

assert u.grad is not None and torch.isfinite(u.grad).all()
print("||grad|| = %.4e sur %d pixels, tous finis" % (u.grad.norm().item(), u.grad.numel()))

hp.mollview(u.grad.detach().cpu().numpy(), nest=True, cmap="coolwarm",
            title="d(loss)/d(pixel)")
plt.show()

||grad|| = 2.5556e-02 sur 12288 pixels, tous finis


## 7. Ciel partiel (`cell_ids`)

Un sous-ensemble de pixels en NESTED : `pbc` bascule à `False`, `nside` doit
être donné explicitement puisqu'il ne peut plus être déduit de `Npix`.

In [15]:
cell_ids = np.arange(NPIX // 4)              # un quart de ciel contigu en NESTED
patch = DataClass(heal_im[cell_ids], nside=NSIDE, cell_ids=cell_ids)

assert patch.pbc is False, "ciel partiel -> pbc False"
assert patch.N0 == (NSIDE,) and patch.nside == NSIDE

wav_patch = patch.get_wavelet_op(J=3, L=L, kernel_size=KERNELSZ)
conv_patch = wav_patch.apply(patch, 0)
down_patch = wav_patch.downsample(conv_patch.modulus(), 1, inplace=False)

print("patch  : Npix =", patch.array.shape[-1], "| pbc =", patch.pbc)
print("conv   :", tuple(conv_patch.array.shape))
print("down   :", tuple(down_patch.array.shape), "| nside =", down_patch.nside)

st_patch = patch.get_ST_op(J=3, L=L).apply(patch)
print("S2 sur le patch :", tuple(st_patch.S2.shape),
      "| finie à %.2f" % torch.isfinite(st_patch.S2).float().mean().item())

patch  : Npix = 3072 | pbc = False
conv   : (4, 3072)
down   : (4, 768) | nside = 16


S2 sur le patch : (1, 1, 1, 3, 4) | finie à 1.00


## 8. NaN : ce qui marche déjà, ce qui manque

Sans masque, les NaN se propagent — comportement identique au noyau planaire
sans `mask_full_res`. L'option `nan_aware_stats=True` fait ignorer les NaN aux
**réductions** (`mean`, `cov`, ...) et au sous-échantillonnage.

Attention : les **convolutions** restent non protégées, donc un pixel NaN
contamine son voisinage à chaque couche. La correction complète, c'est
l'étape 5 du plan (masques de couche par érosion du support du stencil).

In [16]:
nan_map = heal_im.copy()
nan_map[(np.random.rand(200) * NPIX).astype(int)] = np.nan
d_nan = DataClass(nan_map)

st_plain = d_nan.get_ST_op(J=J, L=L).apply(DataClass(nan_map), norm="vanilla")
st_aware = d_nan.get_ST_op(
    J=J, L=L, wavelet_op_kwargs={"nan_aware_stats": True}
).apply(DataClass(nan_map), norm="vanilla")

print("sans protection : S2 finie à %.2f (les NaN contaminent tout)"
      % torch.isfinite(st_plain.S2).float().mean().item())
print("nan_aware_stats : S2 finie à %.2f, S4 finie à %.2f"
      % (torch.isfinite(st_aware.S2).float().mean().item(),
         torch.isfinite(st_aware.S4).float().mean().item()))

# Les coefficients redeviennent finis, mais pas exacts : les convolutions
# étalent chaque NaN sur son voisinage. On mesure le biais résiduel en
# comparant aux coefficients de la carte propre.
print("(%.1f %% des pixels masqués)" % (100 * np.isnan(nan_map).mean()))
ref, got = st.S2.real, st_aware.S2.real
dev = ((got - ref).abs() / ref.abs()).flatten()
print()
print("biais résiduel sur S2 : médiane %.2e, max %.2e"
      % (dev.median().item(), dev.max().item()))
print("-> c'est ce que l'étape 5 (masques de couche) doit supprimer.")

sans protection : S2 finie à 0.00 (les NaN contaminent tout)
nan_aware_stats : S2 finie à 1.00, S4 finie à 0.31
(1.6 % des pixels masqués)

biais résiduel sur S2 : médiane 2.75e-03, max 4.33e-02
-> c'est ce que l'étape 5 (masques de couche) doit supprimer.


## 9. Comparaison d'interface avec le noyau planaire

Vérification programmatique : quels membres publics de `STL_2D_Kernel_Torch` et
de son opérateur manquent encore côté HEALPix ?

In [17]:
from STL_main.STL_2D_Kernel_Torch import (
    STL_2D_Kernel_Torch,
    WaveletOperator2Dkernel_torch,
)

# on inspecte des *instances* : plusieurs membres (array, device, dtype,
# mask_full_res, ...) ne sont posés qu'à la construction
planar = STL_2D_Kernel_Torch(array=np.random.rand(64, 64), pbc=True)
planar_op = planar.get_wavelet_op(J=3, L=L)


def api(obj):
    return {n for n in dir(obj) if not n.startswith("__")}

REQUIS_OPERATEUR = {
    "J", "L", "WType", "device", "dtype", "mask_full_res", "j_to_dg",
    "apply", "mean", "square_mean", "cov", "standardize", "unstandardize",
    "downsample", "_compute_and_store_cross_cov", "_find_mask",
}
REQUIS_DONNEES = {
    "DT", "N0", "dg", "pbc", "conv_history", "array", "device", "dtype",
    "copy", "modulus", "divide", "get_wavelet_op", "get_ST_op", "get_CS_op",
}

manquant_data = REQUIS_DONNEES - api(data)
manquant_op = REQUIS_OPERATEUR - api(wav_op)

print("membres requis absents (classe de données) :", manquant_data or "aucun")
print("membres requis absents (opérateur)         :", manquant_op or "aucun")
assert not manquant_data and not manquant_op

ecart_data = sorted(api(planar) - api(data))
ecart_op = sorted(api(planar_op) - api(wav_op))
print("\nprésents en 2D et pas en HEALPix (attendu : spécificités planaires) :")
print("  données  :", ecart_data or "aucun")
print("  opérateur:", ecart_op or "aucun")

membres requis absents (classe de données) : aucun
membres requis absents (opérateur)         : aucun

présents en 2D et pas en HEALPix (attendu : spécificités planaires) :
  données  : aucun
  opérateur: ['_build_bump_steerable_wavelet_kernel', '_build_morlet_wavelet_kernel', '_build_reweighting_maps_and_scattering_layer_masks', '_build_wavelet_kernel_from_ifft_crop', '_conv2d_circular', '_crop', '_downsample_tensor', '_get_crop_border_size_fully_flexible', '_get_crop_border_size_largest_scale_layer_flexible', '_get_crop_border_size_largest_scale_second_layer', '_get_padding_mode', '_get_smooth_kernel', '_layer1_mask', '_layer2_mask', '_reweighting_maps_smooth', '_reweighting_maps_wav', '_semicomplex_conv2d_circular', '_wav_kernel']


In [18]:
# get_CS_op doit échouer explicitement, pas silencieusement
try:
    data.get_CS_op()
except NotImplementedError as exc:
    print("get_CS_op ->", str(exc)[:110], "...")

# de même pour mask_full_res
try:
    data.get_wavelet_op(J=J, mask_full_res=DataClass(np.isnan(nan_map).astype(float)))
except NotImplementedError as exc:
    print("mask_full_res ->", str(exc)[:110], "...")

get_CS_op -> The HEALPix cross-spectrum operator is not implemented yet. On the sphere the power spectrum is anafast (see f ...
mask_full_res -> mask_full_res is not supported yet by the HEALPix kernel. The spherical counterpart of the planar layer masks  ...


## Bilan

Certifié par ce notebook :

| | état |
|---|---|
| classe de données conforme à `Base_DataClass` | ✅ |
| `divide`, `get_ST_op`, `copy`, `__getitem__`, `modulus` | ✅ |
| convolution ondelettes + `conv_history` | ✅ |
| sous-échantillonnage (signature du 2D) | ✅ |
| `mean` / `square_mean` / `cov` / `standardize` sur l'opérateur | ✅ |
| chaîne S1–S4, batch, canaux, statistiques croisées | ✅ |
| normalisation `store_ref` / `load_ref`, `iso` | ✅ |
| différentiabilité (synthèse) | ✅ |
| ciel partiel via `cell_ids` | ✅ |
| spectre de puissance (`anafast`) | ⛔ étape 4 |
| masques / NaN complets (`mask_full_res`) | ⛔ étape 5 |
| `Synthesis` généralisée | ⛔ étape 7 |

Reste également à aligner le profil radial et la convention d'angle du noyau
sur FOSCAT pour que les coefficients soient directement comparables à
`foscat.scat_cov` (test de non-régression physique, étape 8).